In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from preprocess import TextPreprocessor , create_data_input

ImportError: cannot import name 'create_data_input' from 'preprocess' (d:\Git\ML-Project\preprocess.py)

In [3]:
#df = pd.read_csv('arxiv_dataset/train.csv')
df = pd.read_csv('arxiv_dataset/train.csv')
df.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,paper_id,title,primary_category,authors,first_author,abstract,days_since_submission,recency_score
0,0,0,0,cs-9308101v1,Dynamic Backtracking,cs.AI,M. L. Ginsberg,M. L. Ginsberg,Because of their occasional need to return to ...,12006,0.05
1,1,1,1,cs-9308102v1,A Market-Oriented Programming Environment and ...,cs.AI,M. P. Wellman,M. P. Wellman,Market price systems constitute a well-underst...,12006,0.05
2,2,2,2,cs-9309101v1,An Empirical Analysis of Search in GSAT,cs.AI,I. P. Gent; T. Walsh,I. P. Gent,We describe an extensive study of search in GS...,11975,0.05
3,3,3,3,cs-9311101v1,The Difficulties of Learning Logic Programs wi...,cs.AI,F. Bergadano; D. Gunetti; U. Trinchero,F. Bergadano,As real logic programmers normally use cut (!)...,11914,0.05
4,4,4,4,cs-9311102v1,Software Agents: Completing Patterns and Const...,cs.AI,J. C. Schlimmer; L. A. Hermens,J. C. Schlimmer,To support the goal of allowing users to recor...,11914,0.05


In [4]:
sbert_embeddings = np.load('arxiv_dataset/SBERTEmbeddings_final.npy')

In [5]:
import scipy.sparse
import joblib

tfidf_matrix = scipy.sparse.load_npz('arxiv_dataset/tfidf_matrix.npz')
preprocessor = joblib.load('arxiv_dataset/text_preprocessor.pkl')

In [6]:
tfidf_matrix.shape

(136238, 10000)

In [6]:
# from sklearn.metrics.pairwise import cosine_similarity as cossim
# pair = (0, 0)
# max = -1.0
# j=3
# for i in range(0, 136237):
#     if max < cossim(tfidf_matrix[j], tfidf_matrix[i])[0][0] and i!=j:
#         max = cossim(tfidf_matrix[j], tfidf_matrix[i])[0][0]
#         pair = (j, i)
# print(max, pair)
    

In [7]:
def build_global_index(df):
    paper_ids = df['paper_id'].values
    
    mappings = {i: pid for i, pid in enumerate(paper_ids)}
    reverse_mapping = {pid: i for i, pid in enumerate(paper_ids)}
    
    return mappings, reverse_mapping
mappings, reverse_mapping = build_global_index(df)
reccency_scores = df['recency_score'].values
# Tốc độ tạo chỉ mất khoảng 0.05 giây
paper_to_cat = pd.Series(df['primary_category'].values, index=df['paper_id']).to_dict()


In [8]:
import json
from sklearn.metrics.pairwise import cosine_similarity as cossin
from sklearn.decomposition import PCA
pca = PCA(n_components=384)
def load_user(path_users='UserDataGenerator/synthetic_users.json', vector_path='user_profiles/user_vectors_halflife_90.npz'):
    with open(path_users, 'r') as f:
        data = json.load(f)
    vector_data = np.load(vector_path)
    users_vector = vector_data['vectors']
    users = data['train']
    users_id = [user['user_id'] for user in users]
    target = {user['user_id'] : user['target_paper'] for user in users}
    negative = {user['user_id'] : user['negative_papers'] for user in users}
    history = {user['user_id'] : [x[0] for x in user['train_history']] for user in users}
    return users_id, target, negative, users_vector, history
        

In [9]:
def history_set(user, history):
    his = set()
    for paper in history[user]:
        his.add(paper_to_cat[paper])
    return his

In [10]:
#create dataloader
#cosine_sim (float) recency_score (float) category_match (int)
def create_data_LTR(embeding_matrix = tfidf_matrix, sbert=False):
    if sbert:
        vector_path = 'user_profiles_SBERT/user_vectors_halflife_90.npz'
    else:
        vector_path = 'user_profiles/user_vectors_halflife_90.npz'
    users_id, target, negative, users_vector, history = load_user(vector_path=vector_path)
    print(users_vector.shape)
    print(embeding_matrix.shape)
    data = [] 
    labels = []
    for u, user in enumerate(users_id):
        user_his_cat = history_set(user, history)
        target_paper = target[user]
        cos = cossin(users_vector[u].reshape(1,-1), embeding_matrix[reverse_mapping[target_paper]].reshape(1,-1))[0][0]
        rec = reccency_scores[reverse_mapping[target_paper]]
        cat_match = int(paper_to_cat[target_paper] in user_his_cat)
        data.append(np.array([cos, rec, cat_match]))
        labels.append(1)

        for neg in negative[user]:
            cos = cossin(users_vector[u].reshape(1,-1), embeding_matrix[reverse_mapping[neg]].reshape(1,-1))[0][0]
            rec = reccency_scores[reverse_mapping[neg]]
            cat_match = int(paper_to_cat[neg] in user_his_cat)
            data.append(np.array([cos, rec, cat_match]))
            labels.append(0)
    labels = np.array(labels).astype(np.int32) 
    data_train = np.array(data)
    return labels, data_train


In [11]:
labels, data_tfidf = create_data_LTR()
labels, data_sberts = create_data_LTR(embeding_matrix=sbert_embeddings, sbert=True)

(1000, 10000)
(136238, 10000)
(1000, 384)
(136238, 384)


In [12]:
print(data_sberts[0])

[-0.13794155  0.05184123  0.        ]


In [13]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score

def train(X_train, y_train):
    # -------------------------------------------------------------------------
    # BƯỚC 1: KHỞI TẠO MÔ HÌNH GRADIENT BOOSTING
    # -------------------------------------------------------------------------
    # Cấu hình các siêu tham số (Hyperparameters) chuẩn cho bài toán LTR nhẹ
    ltr_model = GradientBoostingClassifier(
        n_estimators=500,      # Số lượng cây quyết định dựng nối tiếp nhau
        learning_rate=0.1,     # Tốc độ học (shrinkage) để tránh overfit
        max_depth=3,           # Độ sâu tối đa của mỗi cây (phù hợp với 3-5 features)
        random_state=42,       # Cố định seed để kết quả ổn định sau mỗi lần chạy
        verbose=1              # Bật log để xem tiến trình học của các cây
    )

    # -------------------------------------------------------------------------
    # BƯỚC 2: HUẤN LUYỆN (FIT MODEL)
    # -------------------------------------------------------------------------
    print("Bắt đầu thả xích cho Gradient Boosting học luật xếp hạng...")
    # X_train là ma trận dataloader cỡ [N_samples, 3] của ông
    # y_train là mảng nhãn [N_samples] chứa 0 và 1
    ltr_model.fit(X_train, y_train)
    print(" Huấn luyện hoàn tất!")

    # -------------------------------------------------------------------------
    # BƯỚC 3: ĐÁNH GIÁ NHANH ĐỘ CHÍNH XÁC PHÂN LỚP
    # -------------------------------------------------------------------------
    y_pred = ltr_model.predict(X_train)
    print("\n--- KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TRAIN ---")
    print(f"Accuracy Score: {accuracy_score(y_train, y_pred):.4f}")
    print(classification_report(y_train, y_pred))

    # -------------------------------------------------------------------------
    # BƯỚC 4: XEM TẦM ẢNH HƯỞNG CỦA CÁC ĐẶC TRƯNG (FEATURE IMPORTANCE)
    # -------------------------------------------------------------------------
    # Cái này cực kỳ đắt giá để đưa vào báo cáo/slide đồ án
    importances = ltr_model.feature_importances_
    features_names = ['cosine_sim', 'recency_score', 'category_match']

    print("\n--- TRỌNG SỐ TỰ HỌC ĐƯỢC ĐỐI VỚI TỪNG ĐẶC TRƯNG ---")
    for name, imp in zip(features_names, importances):
        print(f"🔹 {name}: {imp*100:.2f}%")
    return ltr_model

In [14]:
test = pd.read_csv('arxiv_dataset/test.csv')
test.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,paper_id,title,primary_category,authors,first_author,abstract,days_since_submission,recency_score
0,0,0,0,2604.26951v1,Turning the TIDE: Cross-Architecture Distillat...,cs.CL,Gongbo Zhang; Wen Wang; Ye Tian; Li Yuan,Gongbo Zhang,Diffusion large language models (dLLMs) offer ...,47,0.711484
1,1,1,1,2604.26231v1,ProMax: Exploring the Potential of LLM-derived...,cs.IR,Yi Zhang; Yiwen Zhang; Kai Zheng; Tong Chen; H...,Yi Zhang,The remarkable text understanding and generati...,47,0.711484
2,2,2,2,2604.26567v1,AirZoo: A Unified Large-Scale Dataset for Grou...,cs.CV,Xiaoya Cheng; Rouwan Wu; Xinyi Liu; Zeyu Cui; ...,Xiaoya Cheng,Despite the rapid progress in data-driven 3D v...,47,0.711484
3,3,3,3,2604.26565v1,"DenseStep2M: A Scalable, Training-Free Pipelin...",cs.CV,Mingji Ge; Qirui Chen; Zeqian Li; Weidi Xie,Mingji Ge,Long-term video understanding requires interpr...,47,0.711484
4,4,4,4,2604.26520v1,3D-LENS: A 3D Lifting-based Elevated Novel-vie...,cs.CV,William Grolleau; Astrid Sabourin; Guillaume L...,William Grolleau,Aerial-Ground Re-Identification (AG-ReID) is c...,47,0.711484


In [15]:
testing_text = create_data_input(test, columns = ['title', 'abstract', 'primary_category'])
print(len(testing_text))
tfidf_test = preprocessor.transform(testing_text)

7701


In [16]:
val_mapping, reverse_val_mapping = build_global_index(test)
val_rec_scores = test['recency_score'].values
val_paper_to_cat = pd.Series(test['primary_category'].values, index=test['paper_id']).to_dict()

In [17]:
# def create_val_LTR(embeding_matrix = tfidf_test):
#     users_id, target, negative, users_vector, history = load_user()
#     data = [] 
#     for u, user in enumerate(users_id):
#         user_his_cat = history_set(user, history)
#         for pred_paper_idx, pred_paper_embedding in enumerate(embeding_matrix): 
#             cos = cossin(users_vector[u].reshape(1,-1), pred_paper_embedding)[0][0]
#             rec = val_rec_scores[pred_paper_idx]
#             cat_match = int(val_paper_to_cat[pred_paper_idx] in user_his_cat)
#             data.append(np.array([cos, rec, cat_match]))
#     data_train = np.array(data)
#     return data_train

In [18]:
def create_val_LTR(embeding_matrix=tfidf_test):
    users_id, target, negative, users_vector, history = load_user()
    
    num_papers = embeding_matrix.shape[0] 
    all_users_features = []
    
    for u, user in enumerate(users_id):
        user_his_cat = history_set(user, history)
        user_tfidf_2d = users_vector[u].reshape(1, -1)
        
        cos_scores = cossin(user_tfidf_2d, embeding_matrix).flatten()
        rec_scores = val_rec_scores 
        
        cat_match_scores = np.zeros(num_papers, dtype=np.int16)
        for idx in range(num_papers):
            if val_paper_to_cat[val_mapping[idx]] in user_his_cat:
                cat_match_scores[idx] = 1
                
        user_features = np.column_stack((cos_scores, rec_scores, cat_match_scores))
        all_users_features.append(user_features)
        
    data_train = np.vstack(all_users_features)
    
    print(" Kích thước ma trận Val LTR cuối cùng:", data_train.shape)
    return data_train
X_pool = create_val_LTR()

 Kích thước ma trận Val LTR cuối cùng: (7701000, 3)


In [19]:
ltr_model_sbert = train(data_sberts, labels)

Bắt đầu thả xích cho Gradient Boosting học luật xếp hạng...
      Iter       Train Loss   Remaining Time 
         1           0.8176            3.14s
         2           0.8154            2.79s
         3           0.8131            2.71s
         4           0.8115            2.86s
         5           0.8102            2.82s
         6           0.8086            2.76s
         7           0.8077            2.69s
         8           0.8065            2.79s
         9           0.8053            2.77s
        10           0.8046            2.71s
        20           0.7964            2.51s
        30           0.7900            2.42s
        40           0.7856            2.29s
        50           0.7799            2.15s
        60           0.7742            2.03s
        70           0.7700            1.95s
        80           0.7655            1.88s
        90           0.7608            1.81s
       100           0.7576            1.76s
       200           0.7206            

In [20]:
# Dự đoán xác suất cho toàn bộ danh sách ứng viên
# ltr_model.predict_proba trả về mảng cỡ [N_candidates, 2] (cột 0 là nhãn 0, cột 1 là nhãn 1)
# Ông chỉ lấy cột [:, 1] tức là xác suất bài báo ăn điểm Positive
ltr_model = train(data_tfidf, labels)
predicted_scores = ltr_model.predict_proba(X_pool)[:, 1]

# Cầm predicted_scores này gắn ngược lại vào danh sách bài báo rồi dùng .sort() từ cao xuống thấp là ra bảng xếp hạng LTR chuẩn chỉ!

Bắt đầu thả xích cho Gradient Boosting học luật xếp hạng...
      Iter       Train Loss   Remaining Time 
         1           0.8175            2.41s
         2           0.8154            2.37s
         3           0.8137            2.52s
         4           0.8120            2.45s
         5           0.8106            2.47s
         6           0.8093            2.43s
         7           0.8080            2.45s
         8           0.8071            2.39s
         9           0.8060            2.33s
        10           0.8052            2.32s
        20           0.7981            2.12s
        30           0.7921            2.00s
        40           0.7865            2.19s
        50           0.7804            2.10s
        60           0.7754            2.00s
        70           0.7703            1.92s
        80           0.7659            1.85s
        90           0.7613            1.82s
       100           0.7566            1.78s
       200           0.7188            

In [21]:
import numpy as np
import pandas as pd

def get_top_k_recommendations(target_user_id, candidate_df, candidate_embeddings, top_k=10):
    """
    Hệ thống gợi ý bài báo thực tế (Production)
    """
    # 1. Load não bộ (Lịch sử & Vector của User)
    users_id, _, _, users_vector, history = load_user(vector_path='user_profiles/user_vectors_halflife_90.npz')
    reverse_user = {u:i for i,u in enumerate(users_id)}
    
    if target_user_id not in users_id:
        return f" Lỗi: User '{target_user_id}' không tồn tại trong hệ thống!"
        
    user_his_cat = history_set(target_user_id, history)
    
    user_vec = users_vector[reverse_user[target_user_id]].reshape(1, -1)
    
    # 2. Xây dựng đặc trưng cho toàn bộ kho ứng viên (7.701 bài)
    cos_scores = cossin(user_vec, candidate_embeddings).flatten()
    rec_scores = candidate_df['recency_score'].values
    
    candidate_categories = candidate_df['primary_category'].values
    cat_match_scores = np.isin(candidate_categories, list(user_his_cat)).astype(np.int16)
    
    X_candidates = np.column_stack((cos_scores, rec_scores, cat_match_scores))
    
    # 3. Phán quyết bằng Gradient Boosting
    predicted_probs = ltr_model.predict_proba(X_candidates)[:, 1]
    
    # 4. Sắp xếp (Sorting) và cắt Top K
    # np.argsort()[::-1] giúp xếp từ điểm cao nhất xuống thấp nhất
    top_k_indices = np.argsort(predicted_probs)[::-1][:top_k]
    
    # 5. Xuất kết quả ra DataFrame cho đẹp
    top_k_papers = candidate_df.iloc[top_k_indices].copy()
    top_k_papers['recommendation_score'] = predicted_probs[top_k_indices]
    
    # Đánh số thứ tự hạng (Rank)
    top_k_papers.insert(0, 'Rank', range(1, top_k + 1))
    
    print(f"\n🚀 TRẢ KẾT QUẢ: TOP {top_k} BÀI BÁO GỢI Ý CHO [{target_user_id}]")
    print(f"Lịch sử chuyên mục yêu thích: {user_his_cat}")
    print("-" * 60)
    
    # Chọn mấy cột quan trọng để hiển thị thôi cho gọn
    return top_k_papers[['Rank', 'paper_id', 'title', 'primary_category', 'recommendation_score']]


# ==========================================
# CÁCH SỬ DỤNG (CHẠY THỬ LUÔN)
# ==========================================
# Lấy thử user đầu tiên trong danh sách để demo
demo_user = "user_0645" 

# Bốc từ file test (7701 bài) kết hợp với embedding SBERT của nó
top_10_df = get_top_k_recommendations(
    target_user_id=demo_user, 
    candidate_df=test,                 # Cái DataFrame test chứa 7701 bài của ông
    candidate_embeddings=tfidf_test, # Ma trận embedding SBERT tương ứng của tập test
    top_k=20
)

# Hiển thị bảng kết quả
display(top_10_df)


🚀 TRẢ KẾT QUẢ: TOP 20 BÀI BÁO GỢI Ý CHO [user_0645]
Lịch sử chuyên mục yêu thích: {'cs.LG', 'cs.CL', 'cs.CV'}
------------------------------------------------------------


,Rank,paper_id,title,primary_category,recommendation_score
1179,1,2604.23568v1,Green-Red Watermarking for Recommender Systems,cs.IR,0.999276
4298,2,2604.14575v1,Generative Augmented Inference,cs.LG,0.999188
3916,3,2604.15830v1,Placing Puzzle Pieces Where They Matter: A Que...,cs.LG,0.999188
4839,4,2604.12807v2,Rethinking Satellite Image Restoration for Onb...,cs.CV,0.998214
1657,5,2604.23018v1,AmaraSpatial-10K: A Spatially and Semantically...,cs.CV,0.997723
3414,6,2604.17504v1,RS-HyRe-R1: A Hybrid Reward Mechanism to Overc...,cs.CV,0.997723
1881,7,2604.21984v2,Soft Anisotropic Diagrams for Differentiable I...,cs.CV,0.997396
4620,8,2604.13398v1,From Prediction to Justification: Aligning Sen...,cs.CL,0.997363
5941,9,2604.02610v1,Structure-Preserving Multi-View Embedding Usin...,stat.ML,0.997030
2237,10,2604.22860v1,Airspeed Forward-Invariance for Unpowered Fixe...,cs.RO,0.997030


In [22]:
def mmr_rerank(candidate_indices, relevance_scores, embeddings, top_k=10, lambda_param=0.7):
    """
    candidate_indices: list các index (trong embeddings) của candidate pool đã prefilter
    relevance_scores: dict hoặc array, relevance_scores[idx] = điểm relevance (lấy từ LTR predict_proba)
    embeddings: ma trận embedding đầy đủ (sbert_embeddings hoặc tfidf_matrix/tfidf_test)
    lambda_param: trọng số relevance vs diversity. lambda=1 -> giống top-K thường, lambda=0 -> chỉ ưu tiên đa dạng
    """
    selected = []
    remaining = list(candidate_indices)

    while len(selected) < top_k and remaining:
        if not selected:
            # bài đầu tiên: chọn relevance cao nhất, chưa có gì để so redundancy
            best_idx = remaining[int(np.argmax([relevance_scores[i] for i in remaining]))]
        else:
            selected_embeds = embeddings[selected]
            mmr_scores = []
            for idx in remaining:
                relevance = relevance_scores[idx]
                sims = cossin(embeddings[idx].reshape(1, -1), selected_embeds)[0]
                redundancy = sims.max()
                mmr_scores.append(lambda_param * relevance - (1 - lambda_param) * redundancy)
            best_idx = remaining[int(np.argmax(mmr_scores))]

        selected.append(best_idx)
        remaining.remove(best_idx)

    return selected

In [23]:
def get_top_k_with_mmr(target_user_id, candidate_df, candidate_embeddings, top_k=10,
                         pool_size=100, lambda_param=0.7, model=None):
    """
    Tái dùng đúng logic tính feature như get_top_k_recommendations,
    nhưng sau khi có predicted_probs thì prefilter pool_size bài tốt nhất,
    rồi áp MMR để re-rank lấy top_k cuối cùng.
    """
    if model is None:
        model = ltr_model  # default dùng model TF-IDF; truyền ltr_model_sbert nếu dùng SBERT

    users_id, _, _, users_vector, history = load_user(vector_path='user_profiles/user_vectors_halflife_90.npz')
    reverse_user = {u: i for i, u in enumerate(users_id)}

    if target_user_id not in users_id:
        return f"Lỗi: User '{target_user_id}' không tồn tại trong hệ thống!"

    user_his_cat = history_set(target_user_id, history)
    user_vec = users_vector[reverse_user[target_user_id]].reshape(1, -1)

    # 1. Tính feature cho toàn bộ candidate (giống get_top_k_recommendations)
    cos_scores = cossin(user_vec, candidate_embeddings).flatten()
    rec_scores = candidate_df['recency_score'].values
    candidate_categories = candidate_df['primary_category'].values
    cat_match_scores = np.isin(candidate_categories, list(user_his_cat)).astype(np.int16)

    X_candidates = np.column_stack((cos_scores, rec_scores, cat_match_scores))
    predicted_probs = model.predict_proba(X_candidates)[:, 1]

    # 2. Prefilter: chỉ giữ pool_size bài có relevance cao nhất làm input cho MMR
    pool_indices = np.argsort(predicted_probs)[::-1][:pool_size]

    # 3. Áp MMR trên pool đã prefilter
    relevance_dict = {idx: predicted_probs[idx] for idx in pool_indices}
    selected_indices = mmr_rerank(
        candidate_indices=list(pool_indices),
        relevance_scores=relevance_dict,
        embeddings=candidate_embeddings,
        top_k=top_k,
        lambda_param=lambda_param
    )

    # 4. Xuất kết quả
    top_k_papers = candidate_df.iloc[selected_indices].copy()
    top_k_papers['recommendation_score'] = predicted_probs[selected_indices]
    top_k_papers.insert(0, 'Rank', range(1, len(selected_indices) + 1))

    print(f"\n🎯 TOP {top_k} BÀI BÁO GỢI Ý (SAU MMR, lambda={lambda_param}) CHO [{target_user_id}]")
    print(f"Lịch sử chuyên mục yêu thích: {user_his_cat}")
    print("-" * 60)

    return top_k_papers[['Rank', 'paper_id', 'title', 'primary_category', 'recommendation_score']]

In [28]:
demo_user = "user_0545"

print("=" * 60)
print("NAIVE TOP-K (không MMR)")
top_k_naive = get_top_k_recommendations(
    target_user_id=demo_user,
    candidate_df=test,
    candidate_embeddings=tfidf_test,
    top_k=10
)
display(top_k_naive)

print("=" * 60)
print("TOP-K SAU MMR")
top_k_mmr = get_top_k_with_mmr(
    target_user_id=demo_user,
    candidate_df=test,
    candidate_embeddings=tfidf_test,
    top_k=10,
    pool_size=100,
    lambda_param=0.7,
    model=ltr_model
)
display(top_k_mmr)

# So sánh nhanh: số category trùng nhau trong top-10
naive_cats = top_k_naive['primary_category'].value_counts()
mmr_cats = top_k_mmr['primary_category'].value_counts()
print("\nPhân bố category — Naive:\n", naive_cats)
print("\nPhân bố category — MMR:\n", mmr_cats)

NAIVE TOP-K (không MMR)

🚀 TRẢ KẾT QUẢ: TOP 10 BÀI BÁO GỢI Ý CHO [user_0545]
Lịch sử chuyên mục yêu thích: {'cs.LG', 'cs.CL', 'cs.NE', 'cs.AI'}
------------------------------------------------------------


,Rank,paper_id,title,primary_category,recommendation_score
4190,1,2604.14866v1,MetaDent: Labeling Clinical Images for Vision-...,cs.CV,0.999914
3510,2,2604.17625v1,FlowC2S: Flowing from Current to Succeeding Fr...,cs.CV,0.999372
7618,3,2510.10990v1,Secret-Protected Evolution for Differentially ...,cs.CR,0.999348
1458,4,2604.22548v1,Multi-output Extreme Spatial Model for Complex...,stat.AP,0.999348
6091,5,2603.28987v1,Multi-fidelity approaches for general constrai...,math.OC,0.999348
5701,6,2604.05329v1,Semantic Trimming and Auxiliary Multi-step Pre...,cs.IR,0.998189
6189,7,2603.25224v1,Fair regression under localized demographic pa...,stat.ML,0.997595
2439,8,2604.19632v1,CreatiParser: Generative Image Parsing of Rast...,cs.CV,0.997595
4420,9,2604.15494v1,ProtoTTA: Prototype-Guided Test-Time Adaptation,cs.LG,0.997399
4911,10,2604.13356v2,Peer-Predictive Self-Training for Language Mod...,cs.CL,0.997396


TOP-K SAU MMR

🎯 TOP 10 BÀI BÁO GỢI Ý (SAU MMR, lambda=0.7) CHO [user_0545]
Lịch sử chuyên mục yêu thích: {'cs.LG', 'cs.CL', 'cs.NE', 'cs.AI'}
------------------------------------------------------------


,Rank,paper_id,title,primary_category,recommendation_score
4190,1,2604.14866v1,MetaDent: Labeling Clinical Images for Vision-...,cs.CV,0.999914
3387,2,2604.17457v3,Beyond the Bellman Fixed Point: Geometry and F...,math.OC,0.997030
3754,3,2604.16767v1,When Misinformation Speaks and Converses: Reth...,cs.CL,0.997363
6812,4,2603.09600v2,A Variational Latent Equilibrium for Learning ...,q-bio.NC,0.994869
6091,5,2603.28987v1,Multi-fidelity approaches for general constrai...,math.OC,0.999348
7618,6,2510.10990v1,Secret-Protected Evolution for Differentially ...,cs.CR,0.999348
553,7,2604.25224v1,ValueAlpha: Agreement-Gated Stress Testing of ...,cs.AI,0.992362
3338,8,2604.18555v1,A Note on TurboQuant and the Earlier DRIVE/EDE...,cs.LG,0.992362
5701,9,2604.05329v1,Semantic Trimming and Auxiliary Multi-step Pre...,cs.IR,0.998189
3190,10,2604.17763v1,A Quasi-Experimental Developer Study of Securi...,cs.CR,0.997030



Phân bố category — Naive:
 primary_category
cs.CV      3
cs.CR      1
stat.AP    1
math.OC    1
cs.IR      1
stat.ML    1
cs.LG      1
cs.CL      1
Name: count, dtype: int64

Phân bố category — MMR:
 primary_category
math.OC     2
cs.CR       2
cs.CV       1
cs.CL       1
q-bio.NC    1
cs.AI       1
cs.LG       1
cs.IR       1
Name: count, dtype: int64


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def compute_ild(selected_indices, embeddings):
    """
    selected_indices: list index (trong embeddings) của Top-K đã chọn
    embeddings: ma trận embedding TƯƠNG ỨNG (phải cùng loại với cái MMR dùng)
    Trả về: ILD = trung bình pairwise dissimilarity (1 - cosine)
    """
    idx = list(selected_indices)
    if len(idx) < 2:
        return 0.0  # 1 paper thì không có "diversity"

    vecs = embeddings[idx]
    # nếu là sparse TF-IDF thì cosine_similarity vẫn nhận được
    sim_matrix = cosine_similarity(vecs)        # K x K
    K = len(idx)

    # lấy phần tam giác trên (i < j), bỏ đường chéo
    iu = np.triu_indices(K, k=1)
    pairwise_sim = sim_matrix[iu]               # vector các sim của cặp i<j
    ild = np.mean(1.0 - pairwise_sim)
    return float(ild)

In [ ]:
def get_indices_naive(target_user_id, candidate_df, candidate_embeddings,
                      top_k=10, model=None):
    """Trả về index của Top-K naive (sort theo predicted_probs)."""
    if model is None:
        model = ltr_model

    users_id, _, _, users_vector, history = load_user(
        vector_path='user_profiles/user_vectors_halflife_90.npz')
    reverse_user = {u: i for i, u in enumerate(users_id)}
    user_his_cat = history_set(target_user_id, history)
    user_vec = users_vector[reverse_user[target_user_id]].reshape(1, -1)

    cos_scores = cossin(user_vec, candidate_embeddings).flatten()
    rec_scores = candidate_df['recency_score'].values
    cat_match = np.isin(candidate_df['primary_category'].values,
                        list(user_his_cat)).astype(np.int16)
    X = np.column_stack((cos_scores, rec_scores, cat_match))
    probs = model.predict_proba(X)[:, 1]

    return list(np.argsort(probs)[::-1][:top_k])


def get_indices_mmr(target_user_id, candidate_df, candidate_embeddings,
                    top_k=10, pool_size=100, lambda_param=0.7, model=None):
    """Trả về index của Top-K sau MMR — tái dùng logic get_top_k_with_mmr."""
    if model is None:
        model = ltr_model

    users_id, _, _, users_vector, history = load_user(
        vector_path='user_profiles/user_vectors_halflife_90.npz')
    reverse_user = {u: i for i, u in enumerate(users_id)}
    user_his_cat = history_set(target_user_id, history)
    user_vec = users_vector[reverse_user[target_user_id]].reshape(1, -1)

    cos_scores = cossin(user_vec, candidate_embeddings).flatten()
    rec_scores = candidate_df['recency_score'].values
    cat_match = np.isin(candidate_df['primary_category'].values,
                        list(user_his_cat)).astype(np.int16)
    X = np.column_stack((cos_scores, rec_scores, cat_match))
    probs = model.predict_proba(X)[:, 1]

    pool_indices = np.argsort(probs)[::-1][:pool_size]
    relevance_dict = {idx: probs[idx] for idx in pool_indices}
    selected = mmr_rerank(
        candidate_indices=list(pool_indices),
        relevance_scores=relevance_dict,
        embeddings=candidate_embeddings,
        top_k=top_k,
        lambda_param=lambda_param
    )
    return selected

In [ ]:
demo_user = "user_0545"
TOP_K = 10

naive_idx = get_indices_naive(demo_user, test, tfidf_test, top_k=TOP_K)
mmr_idx   = get_indices_mmr(demo_user, test, tfidf_test, top_k=TOP_K,
                            pool_size=100, lambda_param=0.7)

ild_naive = compute_ild(naive_idx, tfidf_test)
ild_mmr   = compute_ild(mmr_idx,   tfidf_test)

print(f"ILD Naive Top-K : {ild_naive:.4f}")
print(f"ILD MMR         : {ild_mmr:.4f}")
print(f"Tăng tuyệt đối  : {ild_mmr - ild_naive:+.4f}")
print(f"Tăng tương đối  : {(ild_mmr - ild_naive)/ild_naive*100:+.1f}%")

In [ ]:
def average_ild(user_list, candidate_df, candidate_embeddings,
                top_k=10, lambda_param=0.7):
    naive_vals, mmr_vals = [], []
    for u in user_list:
        try:
            n_idx = get_indices_naive(u, candidate_df, candidate_embeddings, top_k)
            m_idx = get_indices_mmr(u, candidate_df, candidate_embeddings,
                                    top_k, lambda_param=lambda_param)
            naive_vals.append(compute_ild(n_idx, candidate_embeddings))
            mmr_vals.append(compute_ild(m_idx, candidate_embeddings))
        except Exception:
            continue
    return np.mean(naive_vals), np.mean(mmr_vals)

users_id, _, _, _, _ = load_user(
    vector_path='user_profiles/user_vectors_halflife_90.npz')
sample_users = users_id[:50]

avg_naive, avg_mmr = average_ild(sample_users, test, tfidf_test, top_k=10)
print(f"ILD trung bình ({len(sample_users)} users)")
print(f"  Naive : {avg_naive:.4f}")
print(f"  MMR   : {avg_mmr:.4f}  ({(avg_mmr-avg_naive)/avg_naive*100:+.1f}%)")

In [29]:
ltr_model.predict_proba(data_tfidf)[:,1]

array([0.43465006, 0.06347037, 0.08104296, ..., 0.12884267, 0.13413599,
       0.11037886], shape=(7000,))

In [30]:
import time

def evaluate(embedding_matrix, top_k=10):
    users_id, target_dict, _, users_vector, history = load_user()
    num_val_users = len(users_id)
    num_papers_total = embedding_matrix.shape[0]
    
    mrr_list = []          # Lưu MRR toàn cục
    mrr_at_k_list = []     # Lưu MRR@10
    
    print(f"🚀 Bắt đầu đo lường ONLY MRR@{top_k}...")
    start_time = time.time()
    
    # Ép mảng category tĩnh để vector hóa
    all_categories_array = df['primary_category'].values
    
    for i, user in enumerate(users_id):
        user_his_cat = history_set(user, history)
        user_vec = users_vector[i].reshape(1, -1)
        
        # 1. TÍNH ĐIỂM VÀ ĐẶC TRƯNG
        cos_scores = cossin(user_vec, embedding_matrix).flatten()
        rec_scores = reccency_scores 
        cat_match_scores = np.isin(all_categories_array, list(user_his_cat)).astype(np.int16)
        
        user_features = np.column_stack((cos_scores, rec_scores, cat_match_scores))
        
        # 2. DỰ ĐOÁN XÁC SUẤT 
        user_preds = ltr_model.predict_proba(user_features)[:, 1]
        
        # 3. TÌM VỊ TRÍ BÀI TARGET
        target_idx = -1
        real_target_paper = target_dict[user]
        if real_target_paper in reverse_mapping:
            target_idx = reverse_mapping[real_target_paper]
            
        # 4. TÍNH MRR
        if target_idx != -1:
            # Sắp xếp index các bài báo theo xác suất từ cao xuống thấp
            ranked_indices = np.argsort(user_preds)[::-1]
            
            # Khảo sát xem bài Target đang nằm ở vị trí thứ mấy (Cộng 1 vì index bắt đầu từ 0)
            rank = np.where(ranked_indices == target_idx)[0][0] + 1
            
            # Lưu MRR Toàn cục
            mrr_list.append(1.0 / rank)
            
            # Lưu MRR@10 (Ngoài Top 10 = 0 điểm)
            if rank <= top_k:
                mrr_at_k_list.append(1.0 / rank)
            else:
                mrr_at_k_list.append(0.0)
                
        if (i + 1) % 100 == 0:
            print(f"⏳ Đã xử lý {i + 1}/{num_val_users} users... ({(time.time() - start_time):.2f}s)")

    # 5. IN KẾT QUẢ
    print(f"\n✅ KẾT QUẢ ĐÁNH GIÁ MRR TRÊN {len(mrr_list)} USERS:")
    print("-" * 40)
    print(f"🎯 Mean MRR (Toàn cục) : {np.mean(mrr_list):.4f}")
    print(f"🎯 Mean MRR@{top_k}      : {np.mean(mrr_at_k_list):.4f}")
    print("-" * 40)

# --- Kích hoạt chạy với SBERT ---
# evaluate_mrr_only(sbert_embeddings, top_k=10)

In [31]:
# evaluate(tfidf_matrix)
# #evaluate(sbert_embeddings)